In [78]:
from pyspark.sql import SparkSession, Window
from pyspark.sql.functions import countDistinct, sum, month, year, avg, datediff, col, dense_rank, desc
from collections import Counter

In [ ]:
# Initialisation de la session Spark
spark = SparkSession \
    .builder \
    .appName("TradeCorp ETL") \
    .getOrCreate()

print(spark.version);

# Chemin relatif vers les CSV
filepath = "../data/tmp/"

# Création des DataFrames
print(f"Import des DataFrames")
df_categories = spark.read.parquet(filepath + "categories")
df_customers = spark.read.parquet(filepath + "clients")
df_employees = spark.read.parquet(filepath + "employees")
df_orders_details = spark.read.parquet(filepath + "details_commandes")
df_orders = spark.read.parquet(filepath + "commandes")
df_products = spark.read.parquet(filepath + "produits")
df_shippers = spark.read.parquet(filepath + "transporteurs")
df_suppliers = spark.read.parquet(filepath + "fournisseurs")

# Dictionnaire de tout les DataFrames
df_collection = {"categories" : df_categories, 
                 "clients" : df_customers, 
                 "employees" : df_employees, 
                 "details_commandes" : df_orders_details, 
                 "commandes" : df_orders, 
                 "produits" : df_products, 
                 "transporteurs" : df_shippers, 
                 "fournisseurs": df_suppliers}

# Vérification des DataFrames
for name, df in df_collection.items():
    print(f"Vérification du dataframe {name}")
    print(f"Nombre de lignes : {df.count()}")
    df.show(5)
    df.printSchema()
    print("\n")




4.2.0
Import des DataFrames
Vérification du dataframe categories
Nombre de lignes : 8
+-----------+--------------+--------------------+-------+
|category_id| category_name|         description|picture|
+-----------+--------------+--------------------+-------+
|          1|     Beverages|Soft drinks, coff...|   NULL|
|          2|    Condiments|Sweet and savory ...|   NULL|
|          3|   Confections|Desserts, candies...|   NULL|
|          4|Dairy Products|             Cheeses|   NULL|
|          5|Grains/Cereals|Breads, crackers,...|   NULL|
+-----------+--------------+--------------------+-------+
only showing top 5 rows
root
 |-- category_id: integer (nullable = true)
 |-- category_name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- picture: string (nullable = true)



Vérification du dataframe clients
Nombre de lignes : 91
+-----------+--------------------+------------------+--------------------+--------------------+-----------+------+-----------+-------

# Q21 — Jointure orders + customers
Joindre df_orders et df_customers sur customer_id. Garder uniquement : order_id, company_name, country,
order_date, freight.

In [3]:
df_orders_by_company = df_orders \
    .join(df_customers, on="customer_id", how="inner") \
    .select("order_id", "company_name", "country", "order_date", "freight")

df_orders_by_company.show(5)

+--------+--------------------+-------+----------+-------+
|order_id|        company_name|country|order_date|freight|
+--------+--------------------+-------+----------+-------+
|   10400|  Eastern Connection|     UK|1997-01-01|  83.93|
|   10401|Rattlesnake Canyo...|    USA|1997-01-01|  12.51|
|   10402|        Ernst Handel|AUSTRIA|1997-01-02|  67.88|
|   10403|        Ernst Handel|AUSTRIA|1997-01-03|  73.79|
|   10404|Magazzini Aliment...|  ITALY|1997-01-03| 155.97|
+--------+--------------------+-------+----------+-------+
only showing top 5 rows


# Q22 — Jointure order_details + products
Joindre df_order_details et df_products sur product_id. Ajouter les colonnes product_name, category_id,
unit_price depuis products.

In [4]:
df_orders_details_with_products = df_orders_details \
    .join(df_products, on="product_id") \
    .select(df_orders_details["*"], df_products["product_name"], df_products["category_id"], df_products["unit_price"])

df_orders_details_with_products.show(5)

+--------+----------+-------------+--------+--------+----------+--------------------+-----------+----------+
|order_id|product_id|prix_unitaire|quantite|discount|sous_total|        product_name|category_id|unit_price|
+--------+----------+-------------+--------+--------+----------+--------------------+-----------+----------+
|   10248|        11|         14.0|      12|     0.0|     168.0|      Queso Cabrales|          4|      21.0|
|   10248|        72|         34.8|       5|     0.0|     174.0|Mozzarella di Gio...|          4|      34.8|
|   10249|        14|         18.6|       9|     0.0|     167.4|                Tofu|          7|     23.25|
|   10249|        51|         42.4|      40|     0.0|    1696.0|Manjimup Dried Ap...|          7|      53.0|
|   10250|        41|          7.7|      10|     0.0|      77.0|Jack's New Englan...|          8|      9.65|
+--------+----------+-------------+--------+--------+----------+--------------------+-----------+----------+
only showing top 5 

# Q23 — Jointure products + categories
Joindre df_products et df_categories sur category_id pour enrichir chaque produit avec category_name et
description.

In [5]:
df_products_with_categories = df_products \
    .join(df_categories, on="category_id") \
    .select(df_products["*"], "category_name", "description")

df_products_with_categories.show(5)

+----------+--------------------+-----------+-----------+-------------------+----------+--------------+--------------+-------------+------------+--------+-------------+--------------------+
|product_id|        product_name|supplier_id|category_id|  quantity_per_unit|unit_price|units_in_stock|units_on_order|reorder_level|discontinued|en_stock|category_name|         description|
+----------+--------------------+-----------+-----------+-------------------+----------+--------------+--------------+-------------+------------+--------+-------------+--------------------+
|         3|       Aniseed Syrup|          1|          2|12 - 550 ml bottles|      10.0|            13|            70|           25|           0|    true|   Condiments|Sweet and savory ...|
|         4|Chef Anton's Caju...|          2|          2|     48 - 6 oz jars|      22.0|            53|             0|            0|           0|    true|   Condiments|Sweet and savory ...|
|         6|Grandma's Boysenb...|          3|     

# Q24 — DataFrame enrichi complet
A - Réaliser une première jointure complète (order_details, orders, customers, products enrichi avec
categories, employees, shippers) sans renommer aucune colonne. Lister ensuite les colonnes qui apparaissent
en double grâce à Counter.

In [6]:
df_megajoin = df_orders_details \
    .join(df_orders, on="order_id") \
    .join(df_customers, on="customer_id") \
    .join(df_products_with_categories, on="product_id") \
    .join(df_employees, on="employee_id") \
    .join(df_shippers, on="shipper_id")

df_megajoin.show(5)

for name, count in Counter(df_megajoin.columns).items():
    if count > 1:
        print(f"Colonne en double : {name} : {count}")

+----------+-----------+----------+-----------+--------+-------------+--------+--------+----------+----------+-------------+------------+-------+--------------------+---------------+-----------+-----------+----------------+------------+----------+--------------------+------------+--------------------+---------------+-----------+------+-----------+-------+--------------+--------------+--------------------+-----------+-----------+------------------+----------+--------------+--------------+-------------+------------+--------+--------------+--------------------+----------+---------+--------------------+----------+-------+-------+-------------+----------------+--------------+
|shipper_id|employee_id|product_id|customer_id|order_id|prix_unitaire|quantite|discount|sous_total|order_date|required_date|shipped_date|freight|           ship_name|   ship_address|  ship_city|ship_region|ship_postal_code|ship_country|is_shipped|        company_name|contact_name|       contact_title|        address|  

B - Pour chaque colonne identifiée en Q24a, la renommer dans sa table d'origine avant de refaire la jointure,
en la préfixant selon la table (customer_country, employee_country, shipper_name...). Reconstruire ensuite
df_orders_enriched avec ces tables renommées, puis vérifier qu'il ne reste plus aucun doublon.

In [7]:
col_renamed_customers = {"company_name": "customer_company_name", "city" : "customer_city", "country" : "customer_country", "phone" : "customer_phone"}
col_renamed_shippers = {"company_name": "shipper_company_name", "phone" : "shipper_phone"}
col_renamed_employees = {"city" : "employee_city", "country" : "employee_country"}

df_customers = df_customers.withColumnsRenamed(col_renamed_customers)
df_shippers = df_shippers.withColumnsRenamed(col_renamed_shippers)
df_employees = df_employees.withColumnsRenamed(col_renamed_employees)

df_orders_enriched = df_orders_details \
    .join(df_orders, on="order_id") \
    .join(df_customers, on="customer_id") \
    .join(df_products_with_categories, on="product_id") \
    .join(df_employees, on="employee_id") \
    .join(df_shippers, on="shipper_id")

df_orders_enriched.show(5)

for name, count in Counter(df_orders_enriched.columns).items():
    if count > 1:
        print(f"Colonne en double : {name} : {count}")

+----------+-----------+----------+-----------+--------+-------------+--------+--------+----------+----------+-------------+------------+-------+--------------------+---------------+-----------+-----------+----------------+------------+----------+---------------------+------------+--------------------+---------------+-------------+------+-----------+----------------+--------------+--------------+--------------------+-----------+-----------+------------------+----------+--------------+--------------+-------------+------------+--------+--------------+--------------------+----------+---------+--------------------+----------+-------------+----------------+-------------+--------------------+--------------+
|shipper_id|employee_id|product_id|customer_id|order_id|prix_unitaire|quantite|discount|sous_total|order_date|required_date|shipped_date|freight|           ship_name|   ship_address|  ship_city|ship_region|ship_postal_code|ship_country|is_shipped|customer_company_name|contact_name|       

# Q25 — CA par client
Calculer le chiffre d'affaires total par client (company_name) depuis df_orders_enriched. Trier par CA
décroissant. Afficher le top 10.

In [8]:
df_ca_by_company = df_orders_enriched \
        .groupby("customer_company_name") \
        .sum("sous_total") \
        .orderBy("sum(sous_total)", ascending=False)

df_ca_by_company.show(10)


+---------------------+------------------+
|customer_company_name|   sum(sous_total)|
+---------------------+------------------+
|           QUICK-Stop| 51682.73999999999|
|   Save-a-lot Markets|          40238.09|
|         Ernst Handel|          39975.91|
|       Mère Paillarde|22871.070000000003|
| Rattlesnake Canyo...|           17636.1|
|        Simons bistro|          16232.42|
| Hungry Owl All-Ni...|          14403.03|
|       Folk och fä HB|          13200.92|
|     HILARION-Abastos|          11799.74|
|   Berglunds snabbköp|          11758.92|
+---------------------+------------------+
only showing top 10 rows


# Q26 — CA par catégorie
Calculer le CA total par catégorie de produits. Afficher le nombre de produits distincts vendus par catégorie.

In [9]:
df_ca_by_categorie = df_orders_enriched \
        .groupBy("category_name") \
        .agg(countDistinct("product_id").alias("Nb_produit_distinct"), sum("sous_total").alias("CA"))

df_ca_by_categorie.show()

+--------------+-------------------+------------------+
| category_name|Nb_produit_distinct|                CA|
+--------------+-------------------+------------------+
|Dairy Products|                  9|108086.90000000002|
|  Meat/Poultry|                  2|          11017.17|
|    Condiments|                 11|           54995.0|
|     Beverages|                  9|          90368.64|
|Grains/Cereals|                  6|51463.630000000005|
|       Seafood|                 12|          66959.23|
|   Confections|                 13| 82657.78000000001|
|       Produce|                  4|          40992.09|
+--------------+-------------------+------------------+



# Q27 — CA par mois
Calculer le CA mensuel. Utiliser date_trunc ou month() et year() pour extraire le mois et l'année.

In [10]:
df_ca_by_month = df_orders_enriched \
        .groupBy(year("order_date").alias("Annee"), month("order_date").alias("Mois")) \
        .agg(sum("sous_total").alias("CA_Mensuel")) \
        .orderBy("Annee", "Mois")

df_ca_by_month.show()
        

+-----+----+------------------+
|Annee|Mois|        CA_Mensuel|
+-----+----+------------------+
| 1997|   1|51487.509999999995|
| 1997|   2|31549.039999999997|
| 1997|   3|          33226.33|
| 1997|   4| 41510.59999999999|
| 1997|   5|48895.270000000004|
| 1997|   6|29875.469999999998|
| 1997|   7| 45162.87999999999|
| 1997|   8| 38039.92999999999|
| 1997|   9| 43335.42999999999|
| 1997|  10|           48574.5|
| 1997|  11|          39898.78|
| 1997|  12|54984.700000000004|
+-----+----+------------------+



# Q28 — Performance par employé
Calculer pour chaque employé (full_name) : le nombre de commandes traitées, le CA total généré et le délai
moyen de livraison en jours.

In [11]:
df_perf_by_employee = df_orders_enriched \
        .groupBy("full_name") \
        .agg(countDistinct("order_id").alias("Nombre_Commandes"),
             sum("sous_total").alias("CA_Total"),
             avg(datediff(col("shipped_date"), col("order_date"))).alias("Delai_Moyen")
        )

df_perf_by_employee.show()

+----------------+----------------+------------------+------------------+
|       full_name|Nombre_Commandes|          CA_Total|       Delai_Moyen|
+----------------+----------------+------------------+------------------+
|  Anne Dodsworth|              18|20595.989999999998|10.026315789473685|
|   Nancy Davolio|              54|          81898.92| 7.839416058394161|
|   Andrew Fuller|              40|          54907.03| 10.19047619047619|
| Steven Buchanan|              18|          17185.65| 6.461538461538462|
| Janet Leverling|              71|          97081.27| 8.921212121212122|
|     Robert King|              33|49562.780000000006|  9.81081081081081|
|  Laura Callahan|              53|47077.950000000004| 7.951456310679611|
|Margaret Peacock|              75|104193.78000000003| 8.302197802197803|
|  Michael Suyama|              33|          34037.07| 7.943661971830986|
+----------------+----------------+------------------+------------------+



# Q29 — Window functions — Rang
Classer les produits par CA généré avec dense_rank(). Utiliser une Window partitionnée par category_name.

In [ ]:
# Creation de la Fenetre
w = Window.partitionBy("category_name").orderBy(desc("CA"))

df_products_ordered_by_ca = df_orders_enriched \
        .groupBy("category_name", "product_name") \
        .agg(sum("sous_total").alias("CA")) \
        .withColumn("rank", dense_rank().over(w))

df_products_ordered_by_ca.show(10)

+-------------+--------------------+-----------------+----+
|category_name|        product_name|               CA|rank|
+-------------+--------------------+-----------------+----+
|    Beverages|       Côte de Blaye|         49198.09|   1|
|    Beverages|         Ipoh Coffee|          11069.9|   2|
|    Beverages|        Lakkalikööri|           7379.1|   3|
|    Beverages|       Outback Lager|           5468.4|   4|
|    Beverages|      Steeleye Stout|           5274.9|   5|
|    Beverages|Rhönbräu Klosterbier|          4485.55|   6|
|    Beverages|    Chartreuse verte|4475.700000000001|   7|
|    Beverages|       Sasquatch Ale|           2107.0|   8|
|    Beverages|Laughing Lumberja...|            910.0|   9|
|   Condiments|Louisiana Fiery H...|           9373.2|   1|
+-------------+--------------------+-----------------+----+
only showing top 10 rows


# Q30 — Window functions — Cumul
Calculer le CA cumulé par mois (ordre chronologique) avec sum() sur une Window orderBy date.

In [ ]:
w = Window.orderBy("Annee", "Mois")

df_ca_cumul_by_month = df_ca_by_month \
        .withColumn("CA_Cumule", sum("CA_Mensuel").over(w))

df_ca_cumul_by_month.show()

+-----+----+------------------+------------------+
|Annee|Mois|        CA_Mensuel|         CA_cumule|
+-----+----+------------------+------------------+
| 1997|   1|51487.509999999995|51487.509999999995|
| 1997|   2|31549.039999999997| 83036.54999999999|
| 1997|   3|          33226.33|116262.87999999999|
| 1997|   4| 41510.59999999999|157773.47999999998|
| 1997|   5|48895.270000000004|         206668.75|
| 1997|   6|29875.469999999998|         236544.22|
| 1997|   7| 45162.87999999999|          281707.1|
| 1997|   8| 38039.92999999999|319747.02999999997|
| 1997|   9| 43335.42999999999|363082.45999999996|
| 1997|  10|           48574.5|411656.95999999996|
| 1997|  11|          39898.78|         451555.74|
| 1997|  12|54984.700000000004|         506540.44|
+-----+----+------------------+------------------+



# Q31 — Tri et limite
Afficher les 5 produits les plus vendus en quantité (toutes commandes confondues).
Afficher les 3 pays clients (customer_country) qui génèrent le plus de chiffre d'affaires.

In [65]:
df_most_solded_products = df_orders_enriched \
       .groupBy("product_name") \
       .agg(sum("quantite").alias("Quantite_Totale")) \
       .orderBy(desc("Quantite_Totale")) \
       .limit(5)

df_most_solded_products.show()

+--------------------+---------------+
|        product_name|Quantite_Totale|
+--------------------+---------------+
|Gnocchi di nonna ...|            971|
|Raclette Courdavault|            752|
|   Camembert Pierrot|            665|
|Rhönbräu Klosterbier|            630|
| Sir Rodney's Scones|            610|
+--------------------+---------------+



In [72]:
df_most_valuable_countries = df_orders_enriched \
       .groupBy("customer_country") \
       .agg(sum("sous_total").alias("CA")) \
       .orderBy(desc("CA")) \
       .limit(3)

df_most_valuable_countries.show()

+----------------+------------------+
|customer_country|                CA|
+----------------+------------------+
|         GERMANY|100641.29000000002|
|             USA| 90731.70999999998|
|         AUSTRIA|          46559.49|
+----------------+------------------+



# Q32 — Écriture en Parquet
Écrire df_orders_enriched en format Parquet dans /home/jovyan/data/output/orders_enriched.parquet. Utiliser le
mode overwrite.

In [70]:
filePath = "../data/output/"

df_orders_enriched.write.parquet(filePath + "orders_enriched", mode="overwrite")

# Notebook 4 — Écriture Parquet (et chargement PostgreSQL bonus)

## Q33 — Relire le Parquet
Relire le fichier Parquet et vérifier que le nombre de lignes est identique à l'original. Afficher le schema —
observer que les types sont préservés.

In [73]:
pq_orders_enriched = spark.read.parquet(filePath + "orders_enriched")

print(f"Nombre de lignes dans le dataframe initial : {df_orders_enriched.count()}")
print(f"Nombre de ligne dans le parquet : {pq_orders_enriched.count()}")

print("Schema initial")
df_orders_enriched.printSchema()

print("Schema parquet")
pq_orders_enriched.printSchema()

Nombre de lignes dans le dataframe initial : 893
Nombre de ligne dans le parquet : 893
Schema initial
root
 |-- shipper_id: integer (nullable = true)
 |-- employee_id: integer (nullable = true)
 |-- product_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_id: integer (nullable = true)
 |-- prix_unitaire: double (nullable = true)
 |-- quantite: integer (nullable = true)
 |-- discount: double (nullable = true)
 |-- sous_total: double (nullable = true)
 |-- order_date: date (nullable = true)
 |-- required_date: date (nullable = true)
 |-- shipped_date: date (nullable = true)
 |-- freight: double (nullable = true)
 |-- ship_name: string (nullable = true)
 |-- ship_address: string (nullable = true)
 |-- ship_city: string (nullable = true)
 |-- ship_region: string (nullable = true)
 |-- ship_postal_code: string (nullable = true)
 |-- ship_country: string (nullable = true)
 |-- is_shipped: boolean (nullable = true)
 |-- customer_company_name: string (nullabl

## Q34 — Comparer CSV vs Parquet
Comparer la taille du fichier CSV original vs le fichier Parquet. Observer le gain de compression. Expliquer
pourquoi Parquet est plus efficace.

Orders _ parquet : 24ko
Orders _ csv : 99ko

Ratio de 1 pour 4 en faveur de parquet

| Critère | CSV | Parquet |
| --- | --- | --- |
| Format | Texte | Binaire |
| Compression | ❌ Non | ✅ Oui (Snappy/Zstd) |
| Encodage | Naïf | RLE, Dictionnaire, Delta |
| Stockage par | Ligne | Colonne |
| Taille sur disque | Élevée | Réduite (4× moins) |

## Q35 — Partitionnement
Écrire le DataFrame partitionné par country avec partitionBy('country'). Observer la structure des dossiers créés.

In [80]:
df_orders_enriched.write.parquet(filePath + "orders_enriched", mode="overwrite").partitionBy("country")

AttributeError: 'NoneType' object has no attribute 'partitionBy'